In [1]:
!pip install azure-ai-textanalytics azure-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [azure-ai-textanalytics]


In [2]:
API_KEY = "7NX4iZO1GNzqzVqUvD2HsrVjraIAFMdUQn1iJhMGpbvogRcBEzIdJQQJ99CKACBsN54XJ3w3AAAaACOGJSN0"
ENDPOINT = "https://dheeraj-language-service.cognitiveservices.azure.com/"

In [3]:

# ============================================================
# Step 1: Installing required libraries
# ============================================================

from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from unittest.mock import MagicMock


# ============================================================
# Step 2: Configuring Azure AI Language Credentials
# ============================================================
API_KEY = "7NX4iZO1GNzqzVqUvD2HsrVjraIAFMdUQn1iJhMGpbvogRcBEzIdJQQJ99CKACBsN54XJ3w3AAAaACOGJSN0"
ENDPOINT = "https://dheeraj-language-service.cognitiveservices.azure.com/"



# ============================================================
# Step 3: Initializing the Azure AI Language Client
# ============================================================
def authenticate_client(api_key, endpoint):
    """Create and return an authenticated TextAnalyticsClient."""
    credential = AzureKeyCredential(api_key)
    client = TextAnalyticsClient(endpoint=endpoint, credential=credential)
    return client


# ============================================================
# Step 4: Defining the Chatbot Class
# ============================================================
class IntegratedChatbot:
    """A chatbot that performs sentiment analysis using Azure AI Language.

    This class extends a simple rule-based chatbot with cloud-based
    sentiment analysis from Azure AI Language (Text Analytics API).

    Features:
        - Sentiment analysis on user messages
        - Command recognition (help, greetings, farewells, about)
        - Graceful handling of empty/malformed input
        - Confidence scores for transparency
    """

    # Keyword sets for command matching
    GREETINGS = {"hello", "hi", "hey", "greetings", "howdy"}
    FAREWELLS = {"bye", "goodbye", "exit", "quit", "see you"}
    HELP_COMMANDS = {"help", "capabilities", "commands", "what can you do"}

    def __init__(self, client):
        """Initialize the chatbot with an Azure Text Analytics client.

        Args:
            client: An authenticated TextAnalyticsClient instance.
        """
        self.client = client
        self.conversation_active = True

    def get_welcome_message(self):
        """Return the welcome message shown when the bot starts."""
        return (
            "=============================================\n"
            "  Welcome to the Integrated AI Chatbot!\n"
            "=============================================\n"
            "I can analyze the sentiment of your messages\n"
            "using Azure AI Language.\n\n"
            "Type 'help' to see my capabilities.\n"
            "Type 'bye' to exit.\n"
            "============================================="
        )

    def process_message(self, user_input):
        """Process a user message and return the bot's response.

        Args:
            user_input: The raw text input from the user.

        Returns:
            A string containing the bot's response.
        """
        # Handle empty or whitespace-only input
        text = user_input.strip()
        if not text:
            return ("It looks like you sent an empty message. "
                    "Please type something and I will analyze it!")

        normalized = text.lower()

        # Checking for commands first
        if normalized in self.HELP_COMMANDS:
            return self._get_capabilities()
        elif normalized in self.GREETINGS:
            return ("Hello! Nice to meet you. How can I help you today?\n"
                    "Type 'help' to see what I can do.")
        elif normalized in self.FAREWELLS:
            self.conversation_active = False
            return "Goodbye! It was great chatting with you. Have a wonderful day!"
        elif normalized == "about":
            return self._get_about()
        else:
            # Perform sentiment analysis on all other input
            return self._analyze_sentiment(text)

    def _get_capabilities(self):
        """Return a formatted list of the bot's capabilities."""
        return (
            "Here are my capabilities:\n\n"
            "1. Sentiment Analysis - Send me any text and I will\n"
            "   analyze whether it is positive, negative, neutral,\n"
            "   or mixed.\n\n"
            "2. Greetings - Say 'hello', 'hi', or 'hey' to start\n"
            "   a conversation.\n\n"
            "3. Farewell - Say 'bye' or 'goodbye' to end our chat.\n\n"
            "4. About - Type 'about' to learn more about this bot.\n\n"
            "5. Help - Type 'help' or 'capabilities' to see this\n"
            "   list again.\n\n"
            "Try typing a sentence and I will tell you what\n"
            "sentiment I detect!"
        )

    def _get_about(self):
        """Return information about the bot."""
        return (
            "Integrated AI Chatbot\n"
            "---------------------\n"
            "Built using Python and Azure AI Language for\n"
            "sentiment analysis.\n\n"
            "Course: MSAI-631 - AI for Human-Computer Interaction\n"
            "University: University of the Cumberlands\n"
            "Author: Dheeraj Padala\n\n"
            "This bot demonstrates how traditional chatbots can\n"
            "be enhanced with cloud-based AI services."
        )

    def _analyze_sentiment(self, text):
        """Analyze sentiment using Azure AI Language and return a response.

        Args:
            text: The user's message to analyze.

        Returns:
            A formatted string with sentiment results.
        """
        try:
            # Call Azure AI Language API
            response = self.client.analyze_sentiment(documents=[text])
            result = response[0]

            if result.is_error:
                return ("I encountered an error analyzing your message. "
                        "Please try rephrasing it.")

            sentiment = result.sentiment
            confidence = result.confidence_scores

            # Emoji and contextual phrasing for each sentiment
            sentiment_map = {
                "positive": {
                    "emoji": "😊",
                    "response": "That sounds great! I am glad to hear something positive.",
                },
                "negative": {
                    "emoji": "😔",
                    "response": "I am sorry to hear that. I hope things get better.",
                },
                "neutral": {
                    "emoji": "😐",
                    "response": "That seems like a neutral statement.",
                },
                "mixed": {
                    "emoji": "🤔",
                    "response": "Your message has mixed sentiments — both positive and negative elements.",
                },
            }

            info = sentiment_map.get(sentiment, sentiment_map["neutral"])

            # Format confidence scores as percentages
            pos_pct = f"{confidence.positive * 100:.1f}%"
            neg_pct = f"{confidence.negative * 100:.1f}%"
            neu_pct = f"{confidence.neutral * 100:.1f}%"

            return (
                f"Sentiment Analysis Result {info['emoji']}\n"
                f"{'=' * 40}\n"
                f"Your message: \"{text}\"\n\n"
                f"Detected Sentiment: {sentiment.capitalize()}\n\n"
                f"Confidence Scores:\n"
                f"  - Positive: {pos_pct}\n"
                f"  - Negative: {neg_pct}\n"
                f"  - Neutral:  {neu_pct}\n\n"
                f"{info['response']}"
            )

        except Exception as exc:
            return (f"Something went wrong while analyzing your message: {exc}\n"
                    f"Please try again.")


# ============================================================
# Step 5: Testing Azure AI Language Connection
# ============================================================
def test_connection(client):
    """Test the Azure AI Language connection with sample texts."""
    test_texts = [
        "I absolutely love this class! It is very interesting.",
        "I am very frustrated with this error.",
        "The weather is partly cloudy today.",
    ]

    print("Testing Azure AI Language connection...\n")
    response = client.analyze_sentiment(documents=test_texts)

    for i, result in enumerate(response):
        if not result.is_error:
            print(f'Text: "{test_texts[i]}"')
            print(f"  Sentiment: {result.sentiment}")
            print(f"  Positive:  {result.confidence_scores.positive:.2%}")
            print(f"  Negative:  {result.confidence_scores.negative:.2%}")
            print(f"  Neutral:   {result.confidence_scores.neutral:.2%}")
            print()
        else:
            print(f"Error: {result.error}")

    print("Connection test complete!")


# ============================================================
# Step 6: Running Automated Test Messages
# ============================================================
def run_automated_tests(bot):
    """Run a set of test messages through the bot."""
    test_messages = [
        "hello",
        "help",
        "about",
        "",               # empty message test
        "   ",            # whitespace-only test
        "I am having a wonderful day today!",
        "This is terrible, I hate waiting in long lines.",
        "The meeting is scheduled for 3 PM.",
        "I love the food but the service was awful.",
        "goodbye",
    ]

    print("Running automated test messages...\n")
    print("=" * 50)

    for msg in test_messages:
        display_msg = msg if msg.strip() else "(empty)"
        print(f"\nYou: {display_msg}")
        response = bot.process_message(msg)
        print(f"Bot: {response}")
        print("-" * 50)


# ============================================================
# Step 7: Unit Tests
# ============================================================
def run_unit_tests():
    """Run unit tests to verify bot logic without API calls."""
    mock_client = MagicMock()
    test_bot = IntegratedChatbot(mock_client)

    # Test 1: Empty message handling
    result = test_bot.process_message("")
    assert "empty message" in result.lower(), "Failed: empty message test"
    print("PASSED: Empty message handled correctly")

    # Test 2: Whitespace-only message
    result = test_bot.process_message("   ")
    assert "empty message" in result.lower(), "Failed: whitespace test"
    print("PASSED: Whitespace-only message handled correctly")

    # Test 3: Help command
    result = test_bot.process_message("help")
    assert "capabilities" in result.lower(), "Failed: help test"
    print("PASSED: Help command works")

    # Test 4: Greeting
    result = test_bot.process_message("hello")
    assert "hello" in result.lower() or "nice" in result.lower(), "Failed: greeting test"
    print("PASSED: Greeting works")

    # Test 5: About command
    result = test_bot.process_message("about")
    assert "dheeraj" in result.lower() or "msai-631" in result.lower(), "Failed: about test"
    print("PASSED: About command works")

    # Test 6: All keyword sets are lowercase
    for word in test_bot.GREETINGS | test_bot.FAREWELLS | test_bot.HELP_COMMANDS:
        assert word == word.lower(), f"Failed: '{word}' is not lowercase"
    print("PASSED: All keywords are lowercase")

    print("\nAll unit tests passed!")


# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    # Initialize the client
    client = authenticate_client(API_KEY, ENDPOINT)
    print("Azure AI Language client initialized successfully!")
    print(f"Endpoint: {ENDPOINT}\n")

    # Run connection test
    test_connection(client)

    # Run unit tests
    print("\n" + "=" * 50)
    print("UNIT TESTS")
    print("=" * 50)
    run_unit_tests()

    # Run automated tests
    print("\n" + "=" * 50)
    print("AUTOMATED CHATBOT TESTS")
    print("=" * 50)
    bot_test = IntegratedChatbot(client)
    run_automated_tests(bot_test)

    # Start interactive chatbot
    print("\n" + "=" * 50)
    print("INTERACTIVE CHATBOT")
    print("=" * 50 + "\n")

    bot = IntegratedChatbot(client)
    print(bot.get_welcome_message())
    print()

    while bot.conversation_active:
        try:
            user_input = input("You: ")
            response = bot.process_message(user_input)
            print(f"\nBot: {response}\n")
        except KeyboardInterrupt:
            print("\n\nBot: Chat ended. Goodbye!")
            break

    print("\n[Chat session ended]")

Azure AI Language client initialized successfully!
Endpoint: https://dheeraj-language-service.cognitiveservices.azure.com/

Testing Azure AI Language connection...

Text: "I absolutely love this class! It is very interesting."
  Sentiment: positive
  Positive:  98.00%
  Negative:  0.00%
  Neutral:   2.00%

Text: "I am very frustrated with this error."
  Sentiment: negative
  Positive:  0.00%
  Negative:  100.00%
  Neutral:   0.00%

Text: "The weather is partly cloudy today."
  Sentiment: neutral
  Positive:  0.00%
  Negative:  0.00%
  Neutral:   99.00%

Connection test complete!

UNIT TESTS
PASSED: Empty message handled correctly
PASSED: Whitespace-only message handled correctly
PASSED: Help command works
PASSED: Greeting works
PASSED: About command works
PASSED: All keywords are lowercase

All unit tests passed!

AUTOMATED CHATBOT TESTS
Running automated test messages...


You: hello
Bot: Hello! Nice to meet you. How can I help you today?
Type 'help' to see what I can do.
------------